In [1]:
# Import packages and modules
import pandas as pd
import tensorflow_decision_forests as tfdf

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

2026-09-11 07:42:29.021597: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-11 07:42:29.059816: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


2026-09-11 07:42:33.373157: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
# Check the version of TensorFlow Decision Forests
print("Found TensorFlow Decision Forests v" + tfdf.__version__)

Found TensorFlow Decision Forests v1.9.1


In [3]:
datasetPath = "dataset/10p/all/kernel3/"

In [4]:
# Parameters
datasetPath = "dataset/10p/ngtdm/kernel3/"


In [5]:
# train/validation/test = 70/10/20
train_data = pd.read_csv(f"{datasetPath}dataset.train.csv")
validation_data = pd.read_csv(f"{datasetPath}dataset.validation.csv")
test_data = pd.read_csv(f"{datasetPath}dataset.test.csv")

In [6]:
# Convert the dataset into a TensorFlow dataset.
train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
    train_data, label="label"
)         
val_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
    validation_data, label="label"
)
test_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
    test_data, label="label"
)

2026-09-11 07:42:45.943193: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-09-11 07:42:45.944652: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [7]:
%%time

# Train an SVM model with class weight.
svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(
        kernel="rbf",
        probability=True,
        class_weight="balanced",
        random_state=42
    ))
])

svm_model.fit(train_data.drop(columns=["label"]), train_data["label"])

CPU times: user 47.9 s, sys: 151 ms, total: 48 s
Wall time: 48.4 s


Pipeline(steps=[('scaler', StandardScaler()),
                ('svc',
                 SVC(class_weight='balanced', probability=True,
                     random_state=42))])

In [8]:
# Evaluate the model with sklearn
from sklearn.metrics import accuracy_score

X_val = validation_data.drop(columns=["label"])
y_true = validation_data["label"].astype(int).to_numpy()
y_pred = svm_model.predict(X_val)

accuracy = accuracy_score(y_true, y_pred)
print(f"accuracy: {accuracy:.4f}")

accuracy: 0.7777


In [9]:
# Model Summary (sklearn style)
print(svm_model)

Pipeline(steps=[('scaler', StandardScaler()),
                ('svc',
                 SVC(class_weight='balanced', probability=True,
                     random_state=42))])


In [10]:
# Model features
feature_names = train_data.drop(columns=["label"]).columns.tolist()
print("features:")
print(feature_names)

features:
['Busyness', 'Coarseness', 'Complexity', 'Contrast', 'Strength']


In [11]:
X_val = validation_data.drop(columns=["label"])
y_true = validation_data["label"].astype(int).to_numpy()
pos_probs = svm_model.predict_proba(X_val)[:, 1]

from sklearn.metrics import roc_auc_score
ROC_AUC = roc_auc_score(y_true, pos_probs)
print("The ROC AUC score is %.5f" % ROC_AUC )

The ROC AUC score is 0.97353


In [12]:
# Compute binary classification metrics with sklearn
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    matthews_corrcoef,
    log_loss,
    brier_score_loss,
 )

# Get predictions (probabilities or class labels)
X_val = test_data.drop(columns=["label"])
y_true = test_data["label"].astype(int).to_numpy()
pos_probs = svm_model.predict_proba(X_val)[:, 1]
y_pred = (pos_probs >= 0.5).astype(int)

# Core metrics
metrics = {}
metrics["accuracy"] = accuracy_score(y_true, y_pred)
metrics["precision"] = precision_score(y_true, y_pred, zero_division=0)
metrics["recall"] = recall_score(y_true, y_pred, zero_division=0)
metrics["f1"] = f1_score(y_true, y_pred, zero_division=0)
metrics["mcc"] = matthews_corrcoef(y_true, y_pred)

# Probabilistic metrics
metrics["roc_auc"] = roc_auc_score(y_true, pos_probs)
metrics["pr_auc"] = average_precision_score(y_true, pos_probs)
metrics["log_loss"] = log_loss(y_true, pos_probs, labels=[0,1])
metrics["brier_score"] = brier_score_loss(y_true, pos_probs)

# Confusion matrix and detailed report
cm = confusion_matrix(y_true, y_pred, labels=[0,1])
report = classification_report(y_true, y_pred, digits=4)

print("Sklearn binary metrics:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")
print("\nConfusion matrix:\n", cm)
print("\nClassification report:\n", report)

Sklearn binary metrics:
accuracy: 0.9145
precision: 0.8000
recall: 0.1857
f1: 0.3015
mcc: 0.3599
roc_auc: 0.9355
pr_auc: 0.6619
log_loss: 0.2113
brier_score: 0.0614

Confusion matrix:
 [[7970   41]
 [ 719  164]]

Classification report:
               precision    recall  f1-score   support

           0     0.9173    0.9949    0.9545      8011
           1     0.8000    0.1857    0.3015       883

    accuracy                         0.9145      8894
   macro avg     0.8586    0.5903    0.6280      8894
weighted avg     0.9056    0.9145    0.8897      8894

